# QuantRustLib Python Bindings vs Original QuantLibPY

This notebook validates correctness and benchmarks performance between:

- **Original Python library** (`QuantLibPY/src`)
- **Rust bindings** (`quantlib_py`, built with `maturin`)

It covers:

1. Analytical European pricing (Black-Scholes, spot quoted)
2. Analytical European pricing (Black, forward quoted)
3. Monte Carlo European pricing (Black-Scholes)
4. Monte Carlo Asian pricing (Black-Scholes)

> Before running, make sure `quantlib_py` is installed in your active env: `maturin develop --release` from `quantlib-rs/quantlib_py`.


In [1]:
from __future__ import annotations

import importlib
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd


### Check that venv is active

In [2]:
import sys, site
print("Executable:", sys.executable)
print("Prefix:", sys.prefix)
print("Base prefix:", sys.base_prefix)
print("Venv active:", sys.prefix != sys.base_prefix)
print("Site-packages:", site.getsitepackages())


Executable: /home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/bin/python
Prefix: /home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv
Base prefix: /usr
Venv active: True
Site-packages: ['/home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/lib/python3.12/site-packages', '/home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/local/lib/python3.12/dist-packages', '/home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/lib/python3/dist-packages', '/home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/lib/python3.12/dist-packages']


In [3]:
def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'quantlib-rs').exists() and (p / 'QuantLibPY').exists():
            return p
    raise RuntimeError('Could not locate repo root containing quantlib-rs and QuantLibPY.')

REPO_ROOT = find_repo_root()
QUANTRS_ROOT = REPO_ROOT / 'quantlib-rs'
QUANTLIBPY_ROOT = REPO_ROOT / 'QuantLibPY'

if str(QUANTLIBPY_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANTLIBPY_ROOT))

qlpy = importlib.import_module('src')

try:
    import quantlib_py as qlrs
except Exception as exc:
    raise RuntimeError(
        'Failed to import quantlib_py. Build/install it first with: '
        'cd quantlib-rs/quantlib_py && maturin develop --release'
    ) from exc

mod_file = getattr(qlrs, '__file__', None)
if not mod_file:
    raise RuntimeError('quantlib_py has no __file__. Import likely resolved to wrong module.')

ext_suffixes = {'.so', '.pyd', '.dll', '.dylib'}
resolved_ext_file = mod_file
if Path(mod_file).suffix.lower() not in ext_suffixes:
    # Maturin commonly installs a package wrapper (__init__.py) that re-exports
    # symbols from a sibling extension module (quantlib_py/quantlib_py*.so).
    ext_mod = getattr(qlrs, 'quantlib_py', None)
    resolved_ext_file = getattr(ext_mod, '__file__', None)

if not resolved_ext_file or Path(resolved_ext_file).suffix.lower() not in ext_suffixes:
    raise RuntimeError(
        f'quantlib_py did not resolve to an extension module. package_file={mod_file}, '
        f'extension_file={resolved_ext_file}. Ensure maturin-built extension is installed '
        'in this environment.'
    )

print('Repo root        :', REPO_ROOT)
print('QuantLibPY root  :', QUANTLIBPY_ROOT)
print('quantlib_py file :', mod_file)
print('quantlib_py ext  :', resolved_ext_file)
print('QuantLibPY ver   :', getattr(qlpy, '__version__', 'unknown'))


Repo root        : /home/ramonggv/Rust_projects/QuantRustLib
QuantLibPY root  : /home/ramonggv/Rust_projects/QuantRustLib/QuantLibPY
quantlib_py file : /home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/lib/python3.12/site-packages/quantlib_py/__init__.py
quantlib_py ext  : /home/ramonggv/Rust_projects/QuantRustLib/quantlib-rs/.rustqlibvenv/lib/python3.12/site-packages/quantlib_py/quantlib_py.cpython-312-x86_64-linux-gnu.so
QuantLibPY ver   : 0.7.0


## Helper Functions

The wrappers below normalize both APIs so we can compare prices and timings fairly.


In [4]:
VAL_DATE = '2026-02-11'
VAL_DATE_DT = datetime.strptime(VAL_DATE, '%Y-%m-%d').date()
RNG = np.random.default_rng(42)

def expiry_str_from_days(days: int) -> str:
    return (VAL_DATE_DT + timedelta(days=int(days))).isoformat()

def timed(fn, *args, repeat: int = 5, warmup: int = 1, **kwargs):
    for _ in range(warmup):
        fn(*args, **kwargs)
    t0 = time.perf_counter()
    out = None
    for _ in range(repeat):
        out = fn(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) / repeat
    return out, elapsed

def old_analytical_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, valuation_date=VAL_DATE):
    engine = qlpy.AnalyticalEngine()
    model = qlpy.BlackScholesModel(float(sigma), float(r), float(q))
    out = np.empty(len(spots), dtype=np.float64)
    for i in range(len(spots)):
        opt = qlpy.EuropeanOption(
            S=float(spots[i]),
            K=float(strikes[i]),
            expiry_date=expiry_str_from_days(int(expiry_days[i])),
            valuation_date=valuation_date,
            option_type='call' if bool(is_call[i]) else 'put',
            qty=int(qtys[i]),
        )
        out[i] = engine.calculate_price(opt, model)
    return out

def rust_analytical_bs_batch_numpy(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, valuation_date=VAL_DATE, parallel=True):
    return np.asarray(qlrs.price_european_analytical_bs_batch_numpy(
        np.ascontiguousarray(spots, dtype=np.float64),
        np.ascontiguousarray(strikes, dtype=np.float64),
        np.ascontiguousarray(expiry_days, dtype=np.int64),
        np.ascontiguousarray(is_call, dtype=np.bool_),
        np.ascontiguousarray(qtys, dtype=np.int32),
        valuation_date,
        float(sigma),
        float(r),
        float(q),
        bool(parallel),
    ))

def old_analytical_black_batch(forwards, strikes, expiry_days, is_call, qtys, sigma, r, valuation_date=VAL_DATE):
    engine = qlpy.AnalyticalEngine()
    model = qlpy.BlackModel(float(sigma), float(r))
    out = np.empty(len(forwards), dtype=np.float64)
    for i in range(len(forwards)):
        fwd = float(forwards[i])
        opt = qlpy.EuropeanOption(
            S=fwd,
            K=float(strikes[i]),
            expiry_date=expiry_str_from_days(int(expiry_days[i])),
            valuation_date=valuation_date,
            option_type='call' if bool(is_call[i]) else 'put',
            qty=int(qtys[i]),
            F=fwd,
        )
        out[i] = engine.calculate_price(opt, model)
    return out

def rust_analytical_black_batch_numpy(forwards, strikes, expiry_days, is_call, qtys, sigma, r, valuation_date=VAL_DATE, parallel=True):
    return np.asarray(qlrs.price_european_analytical_black_batch_numpy(
        np.ascontiguousarray(forwards, dtype=np.float64),
        np.ascontiguousarray(strikes, dtype=np.float64),
        np.ascontiguousarray(expiry_days, dtype=np.int64),
        np.ascontiguousarray(is_call, dtype=np.bool_),
        np.ascontiguousarray(qtys, dtype=np.int32),
        valuation_date,
        float(sigma),
        float(r),
        bool(parallel),
    ))

def old_mc_european_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, n_paths=100_000, n_steps=252, seed=1234, valuation_date=VAL_DATE):
    engine = qlpy.MonteCarloEngine(n_paths=int(n_paths), n_steps=int(n_steps), seed=int(seed) if seed is not None else None)
    model = qlpy.BlackScholesModel(float(sigma), float(r), float(q))
    out = np.empty(len(spots), dtype=np.float64)
    for i in range(len(spots)):
        opt = qlpy.EuropeanOption(
            S=float(spots[i]),
            K=float(strikes[i]),
            expiry_date=expiry_str_from_days(int(expiry_days[i])),
            valuation_date=valuation_date,
            option_type='call' if bool(is_call[i]) else 'put',
            qty=int(qtys[i]),
        )
        out[i] = engine.calculate_price(opt, model)
    return out

def rust_mc_european_bs_batch_numpy(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, n_paths=100_000, n_steps=252, seed=1234, valuation_date=VAL_DATE, parallel_paths=True, chunk_size=4096, parallel_batch=True):
    return np.asarray(qlrs.price_european_monte_carlo_bs_batch_numpy(
        np.ascontiguousarray(spots, dtype=np.float64),
        np.ascontiguousarray(strikes, dtype=np.float64),
        np.ascontiguousarray(expiry_days, dtype=np.int64),
        np.ascontiguousarray(is_call, dtype=np.bool_),
        np.ascontiguousarray(qtys, dtype=np.int32),
        valuation_date,
        float(sigma),
        float(r),
        float(q),
        int(n_paths),
        int(n_steps),
        None if seed is None else int(seed),
        bool(parallel_paths),
        int(chunk_size),
        bool(parallel_batch),
    ))

def old_mc_asian_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, averaging_type='arithmetic', n_paths=100_000, n_steps=252, seed=1234, valuation_date=VAL_DATE):
    engine = qlpy.MonteCarloEngine(n_paths=int(n_paths), n_steps=int(n_steps), seed=int(seed) if seed is not None else None)
    model = qlpy.BlackScholesModel(float(sigma), float(r), float(q))
    out = np.empty(len(spots), dtype=np.float64)
    for i in range(len(spots)):
        opt = qlpy.AsianOption(
            S=float(spots[i]),
            K=float(strikes[i]),
            expiry_date=expiry_str_from_days(int(expiry_days[i])),
            valuation_date=valuation_date,
            option_type='call' if bool(is_call[i]) else 'put',
            qty=int(qtys[i]),
            averaging_type=averaging_type,
        )
        out[i] = engine.calculate_price(opt, model)
    return out

def rust_mc_asian_bs_batch_numpy(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, averaging_type='arithmetic', n_paths=100_000, n_steps=252, seed=1234, valuation_date=VAL_DATE, parallel_paths=True, chunk_size=4096, parallel_batch=True):
    return np.asarray(qlrs.price_asian_monte_carlo_bs_batch_numpy(
        np.ascontiguousarray(spots, dtype=np.float64),
        np.ascontiguousarray(strikes, dtype=np.float64),
        np.ascontiguousarray(expiry_days, dtype=np.int64),
        np.ascontiguousarray(is_call, dtype=np.bool_),
        np.ascontiguousarray(qtys, dtype=np.int32),
        valuation_date,
        averaging_type,
        float(sigma),
        float(r),
        float(q),
        int(n_paths),
        int(n_steps),
        None if seed is None else int(seed),
        bool(parallel_paths),
        int(chunk_size),
        bool(parallel_batch),
    ))

def stats_table(old_vals, rust_vals, label):
    old_vals = np.asarray(old_vals, dtype=np.float64)
    rust_vals = np.asarray(rust_vals, dtype=np.float64)
    abs_diff = np.abs(rust_vals - old_vals)
    rel_diff = abs_diff / np.maximum(np.abs(old_vals), 1e-12)
    return pd.DataFrame([
        {
            'section': label,
            'n': len(old_vals),
            'mean_old': float(old_vals.mean()),
            'mean_rust': float(rust_vals.mean()),
            'max_abs_diff': float(abs_diff.max()),
            'mean_abs_diff': float(abs_diff.mean()),
            'max_rel_diff': float(rel_diff.max()),
            'mean_rel_diff': float(rel_diff.mean()),
        }
    ])


## 1) Analytical Black-Scholes (Spot) - Correctness


In [5]:
n = 12
spots = RNG.uniform(70, 140, size=n).astype(np.float64)
strikes = RNG.uniform(70, 140, size=n).astype(np.float64)
expiry_days = RNG.integers(30, 720, size=n, endpoint=True, dtype=np.int64)
is_call = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys = RNG.integers(1, 5, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

sigma, r, q = 0.25, 0.03, 0.01

old_vals = old_analytical_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q)
rust_vals = rust_analytical_bs_batch_numpy(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, parallel=False)

df = pd.DataFrame({
    'spot': spots,
    'strike': strikes,
    'expiry_days': expiry_days,
    'is_call': is_call,
    'qty': qtys,
    'old': old_vals,
    'rust': rust_vals,
    'abs_diff': np.abs(rust_vals - old_vals),
})
display(df)

max_abs = float(np.max(np.abs(rust_vals - old_vals)))
print('max_abs_diff =', max_abs)

# Analytical formulas should match closely.
assert max_abs < 1e-10, f'Analytical BS mismatch too large: {max_abs}'


,spot,strike,expiry_days,is_call,qty,old,rust,abs_diff
0,124.176923,115.070558,498,True,3,61.119298,61.119298,0.000000e+00
1,100.721491,127.593313,567,True,1,5.203940,5.203940,7.105427e-15
2,130.101854,101.038994,555,False,4,13.210031,13.210031,0.000000e+00
3,118.815762,85.906711,164,True,3,101.018830,101.018830,4.263256e-14
4,76.592414,108.820935,281,False,2,61.734961,61.734961,0.000000e+00
5,138.293565,74.467208,352,False,2,0.071110,0.071110,1.998401e-15
6,123.279779,127.934182,373,True,3,34.131617,34.131617,0.000000e+00
7,125.024501,114.216508,60,False,4,4.632804,4.632804,2.842171e-14
8,78.967954,123.066142,407,False,5,207.590902,207.590902,5.684342e-14
9,101.527016,94.816818,136,False,3,8.848050,8.848050,2.131628e-14


max_abs_diff = 5.684341886080802e-14


## 2) Analytical Black-Scholes (Spot) - Throughput


In [6]:
n = 25_000
spots = RNG.uniform(60, 160, size=n).astype(np.float64)
strikes = RNG.uniform(60, 160, size=n).astype(np.float64)
expiry_days = RNG.integers(7, 1080, size=n, endpoint=True, dtype=np.int64)
is_call = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys = RNG.integers(1, 4, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

sigma, r, q = 0.22, 0.025, 0.0

old_vals, t_old = timed(old_analytical_bs_batch, spots, strikes, expiry_days, is_call, qtys, sigma, r, q, repeat=3, warmup=1)
rust_vals, t_rust = timed(rust_analytical_bs_batch_numpy, spots, strikes, expiry_days, is_call, qtys, sigma, r, q, repeat=3, warmup=1, parallel=True)

stats = stats_table(old_vals, rust_vals, 'Analytical BS batch')
stats['t_old_s'] = t_old
stats['t_rust_s'] = t_rust
stats['speedup_old_over_rust'] = t_old / t_rust
display(stats)


,section,n,mean_old,mean_rust,max_abs_diff,mean_abs_diff,max_rel_diff,mean_rel_diff,t_old_s,t_rust_s,speedup_old_over_rust
0,Analytical BS batch,25000,49.322579,49.322579,2.842171e-13,9.228379e-15,0.022519,0.000023,3.837654,0.008751,438.558606


## 3) Analytical Black (Forward) - Correctness + Throughput


In [7]:
n = 20_000
forwards = RNG.uniform(60, 160, size=n).astype(np.float64)
strikes = RNG.uniform(60, 160, size=n).astype(np.float64)
expiry_days = RNG.integers(7, 1080, size=n, endpoint=True, dtype=np.int64)
is_call = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys = RNG.integers(1, 4, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

sigma, r = 0.27, 0.035

old_vals, t_old = timed(old_analytical_black_batch, forwards, strikes, expiry_days, is_call, qtys, sigma, r, repeat=3, warmup=1)
rust_vals, t_rust = timed(rust_analytical_black_batch_numpy, forwards, strikes, expiry_days, is_call, qtys, sigma, r, repeat=3, warmup=1, parallel=True)

stats = stats_table(old_vals, rust_vals, 'Analytical Black batch')
stats['t_old_s'] = t_old
stats['t_rust_s'] = t_rust
stats['speedup_old_over_rust'] = t_old / t_rust
display(stats)

max_abs = float(np.max(np.abs(rust_vals - old_vals)))
print('max_abs_diff =', max_abs)
assert max_abs < 1e-10, f'Analytical Black mismatch too large: {max_abs}'


,section,n,mean_old,mean_rust,max_abs_diff,mean_abs_diff,max_rel_diff,mean_rel_diff,t_old_s,t_rust_s,speedup_old_over_rust
0,Analytical Black batch,20000,51.288138,51.288138,1.705303e-13,8.824596e-15,0.011775,0.000011,3.039308,0.005192,585.396193


max_abs_diff = 1.7053025658242404e-13


## 4) Monte Carlo European (Black-Scholes)

Monte Carlo uses different RNG implementations in each library, so exact path parity is not expected.
We compare statistical closeness and timing.


In [8]:
n = 640
spots = RNG.uniform(60, 180, size=n).astype(np.float64)
strikes = RNG.uniform(60, 180, size=n).astype(np.float64)
expiry_days = RNG.integers(30, 540, size=n, endpoint=True, dtype=np.int64)
is_call = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys = RNG.integers(1, 3, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

sigma, r, q = 0.20, 0.03, 0.0
n_paths, n_steps, seed = 25_000, 126, 2026

old_vals, t_old = timed(
    old_mc_european_bs_batch,
    spots, strikes, expiry_days, is_call, qtys, sigma, r, q,
    n_paths, n_steps, seed,
    repeat=1, warmup=0
)

rust_vals, t_rust = timed(
    rust_mc_european_bs_batch_numpy,
    spots, strikes, expiry_days, is_call, qtys, sigma, r, q,
    n_paths, n_steps, seed,
    repeat=1, warmup=0,
    parallel_paths=True,
    chunk_size=4096,
    parallel_batch=True
)

stats = stats_table(old_vals, rust_vals, 'MC European BS batch')
stats['t_old_s'] = t_old
stats['t_rust_s'] = t_rust
stats['speedup_old_over_rust'] = t_old / t_rust
display(stats)


,section,n,mean_old,mean_rust,max_abs_diff,mean_abs_diff,max_rel_diff,mean_rel_diff,t_old_s,t_rust_s,speedup_old_over_rust
0,MC European BS batch,640,45.367835,45.225735,1.961506,0.30877,1.0,0.09898,0.538088,0.115937,4.6412


## 5) Monte Carlo Asian (Black-Scholes, Arithmetic)


In [9]:
n = 48
spots = RNG.uniform(80, 120, size=n).astype(np.float64)
strikes = RNG.uniform(80, 120, size=n).astype(np.float64)
expiry_days = RNG.integers(30, 540, size=n, endpoint=True, dtype=np.int64)
is_call = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys = RNG.integers(1, 3, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

sigma, r, q = 0.22, 0.025, 0.0
n_paths, n_steps, seed = 20_000, 126, 2027

old_vals, t_old = timed(
    old_mc_asian_bs_batch,
    spots, strikes, expiry_days, is_call, qtys, sigma, r, q,
    'arithmetic', n_paths, n_steps, seed,
    repeat=1, warmup=0
)

rust_vals, t_rust = timed(
    rust_mc_asian_bs_batch_numpy,
    spots, strikes, expiry_days, is_call, qtys, sigma, r, q,
    'arithmetic', n_paths, n_steps, seed,
    repeat=1, warmup=0,
    parallel_paths=True,
    chunk_size=4096,
    parallel_batch=True
)

stats = stats_table(old_vals, rust_vals, 'MC Asian BS batch')
stats['t_old_s'] = t_old
stats['t_rust_s'] = t_rust
stats['speedup_old_over_rust'] = t_old / t_rust
display(stats)


,section,n,mean_old,mean_rust,max_abs_diff,mean_abs_diff,max_rel_diff,mean_rel_diff,t_old_s,t_rust_s,speedup_old_over_rust
0,MC Asian BS batch,48,16.218256,16.156563,0.247792,0.062693,0.628906,0.076543,5.001432,0.960146,5.209034


## 6) Optional: Monte Carlo Heston (European, Single Case)

This is a quick sanity check for Heston support in both libraries.


In [10]:
spot, strike, days = 100.0, 100.0, 365
opt_type, qty = 'call', 1
kappa, theta, sigma, rho, v0, r, q = 1.5, 0.04, 0.4, -0.6, 0.04, 0.03, 0.0
n_paths, n_steps, seed = 40_000, 252, 999

old_engine = qlpy.MonteCarloEngine(n_paths=n_paths, n_steps=n_steps, seed=seed)
old_model = qlpy.HestonModel(kappa, theta, sigma, rho, v0, r, q)
old_opt = qlpy.EuropeanOption(
    S=spot, K=strike, expiry_date=expiry_str_from_days(days),
    valuation_date=VAL_DATE, option_type=opt_type, qty=qty
)
old_price = old_engine.calculate_price(old_opt, old_model)

rust_price = qlrs.price_european_monte_carlo_heston(
    spot=spot, strike=strike, expiry_date=expiry_str_from_days(days),
    option_type=opt_type, qty=qty,
    kappa=kappa, theta=theta, sigma=sigma, rho=rho, v0=v0, r=r, q=q,
    n_paths=n_paths, n_steps=n_steps, seed=seed,
    parallel=True, chunk_size=4096, valuation_date=VAL_DATE
)

print('old_price :', old_price)
print('rust_price:', rust_price)
print('abs_diff  :', abs(rust_price - old_price))


old_price : 9.011349127039058
rust_price: 9.11216463353243
abs_diff  : 0.10081550649337245


## 7) FFT (Black-Scholes, Black, Heston) - Detailed Parity


In [11]:
FFT_CONFIGS = [
    {'n_fft': 2**11, 'b_fft': 160.0, 'interp': 'linear'},
    {'n_fft': 2**12, 'b_fft': 200.0, 'interp': 'cubic'},
]


def _option_types_from_is_call(is_call):
    return ['call' if bool(v) else 'put' for v in np.asarray(is_call, dtype=np.bool_)]


def _expiry_dates_from_days(expiry_days):
    return [expiry_str_from_days(int(d)) for d in np.asarray(expiry_days, dtype=np.int64)]


def old_fft_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE):
    engine = qlpy.FFTEngine(N=int(n_fft), B=float(b_fft), interp=interp)
    model = qlpy.BlackScholesModel(float(sigma), float(r), float(q))
    option_types = _option_types_from_is_call(is_call)
    expiry_dates = _expiry_dates_from_days(expiry_days)

    out = np.empty(len(spots), dtype=np.float64)
    for i in range(len(spots)):
        opt = qlpy.EuropeanOption(
            S=float(spots[i]),
            K=float(strikes[i]),
            expiry_date=expiry_dates[i],
            valuation_date=valuation_date,
            option_type=option_types[i],
            qty=int(qtys[i]),
        )
        out[i] = engine.calculate_price(opt, model)
    return out


def rust_fft_bs_batch(spots, strikes, expiry_days, is_call, qtys, sigma, r, q, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE, parallel=True):
    return np.asarray(qlrs.price_european_fft_bs_batch(
        np.asarray(spots, dtype=np.float64).tolist(),
        np.asarray(strikes, dtype=np.float64).tolist(),
        _expiry_dates_from_days(expiry_days),
        valuation_date,
        _option_types_from_is_call(is_call),
        np.asarray(qtys, dtype=np.int32).tolist(),
        float(sigma),
        float(r),
        float(q),
        int(n_fft),
        float(b_fft),
        interp,
        bool(parallel),
    ), dtype=np.float64)


def old_fft_black_batch(forwards, strikes, expiry_days, is_call, qtys, sigma, r, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE):
    engine = qlpy.FFTEngine(N=int(n_fft), B=float(b_fft), interp=interp)
    model = qlpy.BlackModel(float(sigma), float(r))
    option_types = _option_types_from_is_call(is_call)
    expiry_dates = _expiry_dates_from_days(expiry_days)

    out = np.empty(len(forwards), dtype=np.float64)
    for i in range(len(forwards)):
        fwd = float(forwards[i])
        opt = qlpy.EuropeanOption(
            S=fwd,
            K=float(strikes[i]),
            expiry_date=expiry_dates[i],
            valuation_date=valuation_date,
            option_type=option_types[i],
            qty=int(qtys[i]),
            F=fwd,
        )
        out[i] = engine.calculate_price(opt, model)
    return out


def rust_fft_black_batch(forwards, strikes, expiry_days, is_call, qtys, sigma, r, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE, parallel=True):
    return np.asarray(qlrs.price_european_fft_black_batch(
        np.asarray(forwards, dtype=np.float64).tolist(),
        np.asarray(strikes, dtype=np.float64).tolist(),
        _expiry_dates_from_days(expiry_days),
        valuation_date,
        _option_types_from_is_call(is_call),
        np.asarray(qtys, dtype=np.int32).tolist(),
        float(sigma),
        float(r),
        int(n_fft),
        float(b_fft),
        interp,
        bool(parallel),
    ), dtype=np.float64)


def old_fft_heston_batch(spots, strikes, expiry_days, is_call, qtys, kappa, theta, sigma, rho, v0, r, q, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE):
    engine = qlpy.FFTEngine(N=int(n_fft), B=float(b_fft), interp=interp)
    model = qlpy.HestonModel(float(kappa), float(theta), float(sigma), float(rho), float(v0), float(r), float(q))
    option_types = _option_types_from_is_call(is_call)
    expiry_dates = _expiry_dates_from_days(expiry_days)

    out = np.empty(len(spots), dtype=np.float64)
    for i in range(len(spots)):
        opt = qlpy.EuropeanOption(
            S=float(spots[i]),
            K=float(strikes[i]),
            expiry_date=expiry_dates[i],
            valuation_date=valuation_date,
            option_type=option_types[i],
            qty=int(qtys[i]),
        )
        out[i] = engine.calculate_price(opt, model)
    return out


def rust_fft_heston_batch(spots, strikes, expiry_days, is_call, qtys, kappa, theta, sigma, rho, v0, r, q, n_fft, b_fft, interp='cubic', valuation_date=VAL_DATE, parallel=True):
    return np.asarray(qlrs.price_european_fft_heston_batch(
        np.asarray(spots, dtype=np.float64).tolist(),
        np.asarray(strikes, dtype=np.float64).tolist(),
        _expiry_dates_from_days(expiry_days),
        valuation_date,
        _option_types_from_is_call(is_call),
        np.asarray(qtys, dtype=np.int32).tolist(),
        float(kappa),
        float(theta),
        float(sigma),
        float(rho),
        float(v0),
        float(r),
        float(q),
        int(n_fft),
        float(b_fft),
        interp,
        bool(parallel),
    ), dtype=np.float64)


def build_fft_result_frame(
    model_name,
    underlier_name,
    underliers,
    strikes,
    expiry_days,
    is_call,
    qtys,
    old_vals,
    rust_vals,
    cfg,
    t_old_s=None,
    t_rust_s=None,
):
    underliers = np.asarray(underliers, dtype=np.float64)
    strikes = np.asarray(strikes, dtype=np.float64)
    old_vals = np.asarray(old_vals, dtype=np.float64)
    rust_vals = np.asarray(rust_vals, dtype=np.float64)

    df = pd.DataFrame({
        'underlier': underliers,
        'underlier_type': underlier_name,
        'strike': strikes,
        'expiry_days': np.asarray(expiry_days, dtype=np.int64),
        'is_call': np.asarray(is_call, dtype=np.bool_),
        'qty': np.asarray(qtys, dtype=np.int32),
        'old': old_vals,
        'rust': rust_vals,
    })
    df['model'] = model_name
    df['interp'] = cfg['interp']
    df['n_fft'] = int(cfg['n_fft'])
    df['b_fft'] = float(cfg['b_fft'])
    df['moneyness'] = underliers / np.maximum(strikes, 1e-12)
    df['abs_diff'] = np.abs(df['rust'] - df['old'])
    df['rel_diff'] = df['abs_diff'] / np.maximum(np.abs(df['old']), 1e-8)
    df['moneyness_bucket'] = pd.cut(
        df['moneyness'],
        bins=[0.0, 0.85, 0.95, 1.05, 1.15, np.inf],
        labels=['<0.85', '0.85-0.95', '0.95-1.05', '1.05-1.15', '>1.15'],
        include_lowest=True,
    )

    if t_old_s is not None and t_rust_s is not None:
        t_old_s = float(t_old_s)
        t_rust_s = float(t_rust_s)
        df['t_old_s'] = t_old_s
        df['t_rust_s'] = t_rust_s
        df['t_diff_s'] = t_old_s - t_rust_s
        df['speedup_old_over_rust'] = t_old_s / max(t_rust_s, 1e-12)
        df['rust_faster'] = t_rust_s < t_old_s

    return df


def summarize_fft_results(df):
    agg = {
        'n': ('old', 'size'),
        'mean_old': ('old', 'mean'),
        'mean_rust': ('rust', 'mean'),
        'max_abs_diff': ('abs_diff', 'max'),
        'p95_abs_diff': ('abs_diff', lambda x: float(np.quantile(x, 0.95))),
        'mean_abs_diff': ('abs_diff', 'mean'),
        'max_rel_diff': ('rel_diff', 'max'),
        'mean_rel_diff': ('rel_diff', 'mean'),
    }

    if 't_old_s' in df.columns and 't_rust_s' in df.columns:
        agg.update({
            't_old_s': ('t_old_s', 'first'),
            't_rust_s': ('t_rust_s', 'first'),
            't_diff_s': ('t_diff_s', 'first'),
            'speedup_old_over_rust': ('speedup_old_over_rust', 'first'),
            'rust_faster': ('rust_faster', 'first'),
        })

    return (
        df.groupby(['model', 'interp', 'n_fft', 'b_fft', 'is_call'])
        .agg(**agg)
        .reset_index()
        .sort_values(['model', 'interp', 'n_fft', 'is_call'])
    )


def assert_fft_parity(df, tolerances):
    for model_name, limits in tolerances.items():
        sub = df[df['model'] == model_name]
        max_abs = float(sub['abs_diff'].max())
        p95_abs = float(np.quantile(sub['abs_diff'], 0.95))
        rel_mask = np.abs(sub['old']) > 1e-4
        max_rel = float(sub.loc[rel_mask, 'rel_diff'].max()) if rel_mask.any() else 0.0
        print(
            f"{model_name}: max_abs={max_abs:.8f}, p95_abs={p95_abs:.8f}, max_rel_filtered={max_rel:.8f}"
        )
        assert max_abs < limits['max_abs'], (
            f"{model_name} FFT max_abs too high: {max_abs} >= {limits['max_abs']}"
        )
        assert p95_abs < limits['p95_abs'], (
            f"{model_name} FFT p95_abs too high: {p95_abs} >= {limits['p95_abs']}"
        )
        assert max_rel < limits['max_rel'], (
            f"{model_name} FFT max_rel too high: {max_rel} >= {limits['max_rel']}"
        )



In [12]:
n = 24

# Shared spot dataset for BS + Heston FFT comparisons.
spots = RNG.uniform(55, 180, size=n).astype(np.float64)
strikes_spot = RNG.uniform(55, 180, size=n).astype(np.float64)
expiry_days_spot = RNG.choice(np.array([14, 30, 60, 90, 180, 365, 540, 720], dtype=np.int64), size=n).astype(np.int64)
is_call_spot = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys_spot = RNG.integers(1, 5, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

# Inject deterministic edge-ish cases.
spots[:6] = np.array([60.0, 80.0, 100.0, 120.0, 140.0, 170.0], dtype=np.float64)
strikes_spot[:6] = np.array([140.0, 120.0, 100.0, 80.0, 60.0, 100.0], dtype=np.float64)
expiry_days_spot[:6] = np.array([14, 30, 90, 180, 365, 720], dtype=np.int64)
is_call_spot[:6] = np.array([True, False, True, False, True, False], dtype=np.bool_)
qtys_spot[:6] = np.array([1, 1, 2, 2, 3, 1], dtype=np.int32)

# Forward dataset for Black FFT comparison.
forwards = RNG.uniform(55, 180, size=n).astype(np.float64)
strikes_fwd = RNG.uniform(55, 180, size=n).astype(np.float64)
expiry_days_fwd = RNG.choice(np.array([14, 30, 60, 90, 180, 365, 540, 720], dtype=np.int64), size=n).astype(np.int64)
is_call_fwd = RNG.integers(0, 2, size=n, dtype=np.int64).astype(np.bool_)
qtys_fwd = RNG.integers(1, 5, size=n, endpoint=True, dtype=np.int64).astype(np.int32)

forwards[:6] = np.array([60.0, 82.0, 100.0, 122.0, 145.0, 175.0], dtype=np.float64)
strikes_fwd[:6] = np.array([140.0, 120.0, 100.0, 80.0, 60.0, 100.0], dtype=np.float64)
expiry_days_fwd[:6] = np.array([14, 30, 90, 180, 365, 720], dtype=np.int64)
is_call_fwd[:6] = np.array([True, False, True, False, True, False], dtype=np.bool_)
qtys_fwd[:6] = np.array([1, 1, 2, 2, 3, 1], dtype=np.int32)

# Model parameters.
bs_sigma, bs_r, bs_q = 0.24, 0.03, 0.01
black_sigma, black_r = 0.28, 0.035
heston_kappa, heston_theta, heston_sigma, heston_rho, heston_v0, heston_r, heston_q = 1.7, 0.04, 0.45, -0.65, 0.035, 0.03, 0.01

FFT_TIMING_REPEAT = 3
FFT_TIMING_WARMUP = 1

frames = []
for cfg in FFT_CONFIGS:
    old_bs, t_old_bs = timed(
        old_fft_bs_batch,
        spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        bs_sigma, bs_r, bs_q,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'],
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    rust_bs, t_rust_bs = timed(
        rust_fft_bs_batch,
        spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        bs_sigma, bs_r, bs_q,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'], parallel=True,
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    frames.append(build_fft_result_frame(
        'BS', 'spot', spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        old_bs, rust_bs, cfg, t_old_s=t_old_bs, t_rust_s=t_rust_bs
    ))

    old_black, t_old_black = timed(
        old_fft_black_batch,
        forwards, strikes_fwd, expiry_days_fwd, is_call_fwd, qtys_fwd,
        black_sigma, black_r,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'],
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    rust_black, t_rust_black = timed(
        rust_fft_black_batch,
        forwards, strikes_fwd, expiry_days_fwd, is_call_fwd, qtys_fwd,
        black_sigma, black_r,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'], parallel=True,
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    frames.append(build_fft_result_frame(
        'Black', 'forward', forwards, strikes_fwd, expiry_days_fwd, is_call_fwd, qtys_fwd,
        old_black, rust_black, cfg, t_old_s=t_old_black, t_rust_s=t_rust_black
    ))

    old_heston, t_old_heston = timed(
        old_fft_heston_batch,
        spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        heston_kappa, heston_theta, heston_sigma, heston_rho, heston_v0, heston_r, heston_q,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'],
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    rust_heston, t_rust_heston = timed(
        rust_fft_heston_batch,
        spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        heston_kappa, heston_theta, heston_sigma, heston_rho, heston_v0, heston_r, heston_q,
        n_fft=cfg['n_fft'], b_fft=cfg['b_fft'], interp=cfg['interp'], parallel=True,
        repeat=FFT_TIMING_REPEAT, warmup=FFT_TIMING_WARMUP,
    )
    frames.append(build_fft_result_frame(
        'Heston', 'spot', spots, strikes_spot, expiry_days_spot, is_call_spot, qtys_spot,
        old_heston, rust_heston, cfg, t_old_s=t_old_heston, t_rust_s=t_rust_heston
    ))

fft_results = pd.concat(frames, ignore_index=True)

print('Detailed FFT comparison rows:', len(fft_results))
display(fft_results.head(12))

summary = summarize_fft_results(fft_results)
display(summary)

fft_timing_summary = (
    fft_results
    .groupby(['model', 'interp', 'n_fft', 'b_fft'])
    .agg(
        t_old_s=('t_old_s', 'first'),
        t_rust_s=('t_rust_s', 'first'),
        t_diff_s=('t_diff_s', 'first'),
        speedup_old_over_rust=('speedup_old_over_rust', 'first'),
        rust_faster=('rust_faster', 'first'),
    )
    .reset_index()
    .sort_values(['model', 'interp', 'n_fft'])
)
display(fft_timing_summary)

not_faster = fft_timing_summary[~fft_timing_summary['rust_faster']]
if not_faster.empty:
    print('FFT speed check: Rust is faster in all tested configurations.')
else:
    print('FFT speed check: Rust is slower in some configurations (see table below).')
    display(not_faster)

bucket_summary = (
    fft_results
    .groupby(['model', 'interp', 'moneyness_bucket'])
    .agg(
        n=('old', 'size'),
        max_abs_diff=('abs_diff', 'max'),
        p95_abs_diff=('abs_diff', lambda x: float(np.quantile(x, 0.95))),
        mean_abs_diff=('abs_diff', 'mean'),
        max_rel_diff=('rel_diff', 'max'),
    )
    .reset_index()
    .sort_values(['model', 'interp', 'moneyness_bucket'])
)
display(bucket_summary)

worst_cases = fft_results.sort_values('abs_diff', ascending=False).head(20)
display(
    worst_cases[
        ['model', 'interp', 'n_fft', 'b_fft', 'underlier_type', 'underlier', 'is_call', 'qty', 'expiry_days', 'moneyness', 'old', 'rust', 'abs_diff', 'rel_diff']
    ]
)

# Integration-level tolerances to capture interpolation/implementation differences.
FFT_TOLERANCES = {
    'BS': {'max_abs': 5e-2, 'p95_abs': 2e-2, 'max_rel': 1.0e-1},
    'Black': {'max_abs': 5e-2, 'p95_abs': 2e-2, 'max_rel': 1.0e-1},
    'Heston': {'max_abs': 2.0e-1, 'p95_abs': 8e-2, 'max_rel': 2.5e-1},
}

assert_fft_parity(fft_results, FFT_TOLERANCES)



Detailed FFT comparison rows: 144


,underlier,underlier_type,strike,expiry_days,is_call,qty,old,rust,model,interp,...,b_fft,moneyness,abs_diff,rel_diff,moneyness_bucket,t_old_s,t_rust_s,t_diff_s,speedup_old_over_rust,rust_faster
0,60.000000,spot,140.000000,14,True,1,-0.002826,-0.002826,BS,linear,...,160.0,0.428571,0.000000e+00,0.000000e+00,<0.85,0.011401,0.002357,0.009043,4.836184,True
1,80.000000,spot,120.000000,30,False,1,39.766828,39.766828,BS,linear,...,160.0,0.666667,1.421085e-14,3.573545e-16,<0.85,0.011401,0.002357,0.009043,4.836184,True
2,100.000000,spot,100.000000,90,True,2,9.955275,9.955275,BS,linear,...,160.0,1.000000,5.684342e-14,5.709879e-15,0.95-1.05,0.011401,0.002357,0.009043,4.836184,True
3,120.000000,spot,80.000000,180,False,2,0.069803,0.069803,BS,linear,...,160.0,1.500000,2.842171e-14,4.071714e-13,>1.15,0.011401,0.002357,0.009043,4.836184,True
4,140.000000,spot,60.000000,365,True,3,241.135008,241.135008,BS,linear,...,160.0,2.333333,0.000000e+00,0.000000e+00,>1.15,0.011401,0.002357,0.009043,4.836184,True
5,170.000000,spot,100.000000,720,False,1,0.783195,0.783195,BS,linear,...,160.0,1.700000,1.421085e-14,1.814472e-14,>1.15,0.011401,0.002357,0.009043,4.836184,True
6,141.630828,spot,95.203002,365,False,3,1.360346,1.360346,BS,linear,...,160.0,1.487672,4.263256e-14,3.133949e-14,>1.15,0.011401,0.002357,0.009043,4.836184,True
7,118.071862,spot,175.063662,365,False,3,161.570346,161.570346,BS,linear,...,160.0,0.674451,2.842171e-14,1.759092e-16,<0.85,0.011401,0.002357,0.009043,4.836184,True
8,70.074750,spot,72.428542,720,True,5,46.971255,46.971255,BS,linear,...,160.0,0.967502,3.552714e-14,7.563591e-16,0.95-1.05,0.011401,0.002357,0.009043,4.836184,True
9,86.426141,spot,150.846007,90,False,4,254.074036,254.074036,BS,linear,...,160.0,0.572943,1.705303e-13,6.711833e-16,<0.85,0.011401,0.002357,0.009043,4.836184,True


,model,interp,n_fft,b_fft,is_call,n,mean_old,mean_rust,max_abs_diff,p95_abs_diff,mean_abs_diff,max_rel_diff,mean_rel_diff,t_old_s,t_rust_s,t_diff_s,speedup_old_over_rust,rust_faster
0,BS,cubic,4096,200.0,False,13,8.544228e+01,8.544228e+01,1.421085e-13,1.250555e-13,3.607371e-14,1.167821e-08,8.983875e-10,0.042980,0.003411,0.039570,12.600770,True
1,BS,cubic,4096,200.0,True,11,5.591316e+01,5.591316e+01,1.421085e-13,1.350031e-13,6.653264e-14,2.842171e-06,2.583792e-07,0.042980,0.003411,0.039570,12.600770,True
2,BS,linear,2048,160.0,False,13,8.544138e+01,8.544138e+01,1.705303e-13,1.023182e-13,3.716685e-14,4.685596e-11,3.655243e-12,0.011401,0.002357,0.009043,4.836184,True
3,BS,linear,2048,160.0,True,11,5.594129e+01,5.594129e+01,1.421085e-13,9.947598e-14,3.552714e-14,5.709879e-15,1.542963e-15,0.011401,0.002357,0.009043,4.836184,True
4,Black,cubic,4096,200.0,False,14,3.078138e-08,3.078059e-08,2.238210e-12,1.707079e-12,7.922234e-13,2.238210e-04,4.179920e-05,0.067556,0.003489,0.064066,19.360818,True
5,Black,cubic,4096,200.0,True,10,7.031621e+00,7.031621e+00,1.307399e-12,1.227463e-12,6.626189e-13,3.663073e-05,3.663073e-06,0.067556,0.003489,0.064066,19.360818,True
6,Black,linear,2048,160.0,False,14,-1.117732e-02,-1.117732e-02,1.278977e-12,1.001865e-12,3.592047e-13,7.703765e-11,3.136988e-11,0.010472,0.002104,0.008368,4.976450,True
7,Black,linear,2048,160.0,True,10,7.024355e+00,7.024355e+00,1.413980e-12,1.033484e-12,3.430945e-13,9.022613e-09,9.022658e-10,0.010472,0.002104,0.008368,4.976450,True
8,Heston,cubic,4096,200.0,False,13,8.514124e+01,8.514124e+01,1.136868e-13,9.663381e-14,4.481885e-14,6.063101e-12,5.005625e-13,0.072264,0.013468,0.058796,5.365686,True
9,Heston,cubic,4096,200.0,True,11,5.304560e+01,5.304560e+01,1.421085e-13,9.947598e-14,3.229740e-14,2.842171e-06,2.583792e-07,0.072264,0.013468,0.058796,5.365686,True


,model,interp,n_fft,b_fft,t_old_s,t_rust_s,t_diff_s,speedup_old_over_rust,rust_faster
0,BS,cubic,4096,200.0,0.042980,0.003411,0.039570,12.600770,True
1,BS,linear,2048,160.0,0.011401,0.002357,0.009043,4.836184,True
2,Black,cubic,4096,200.0,0.067556,0.003489,0.064066,19.360818,True
3,Black,linear,2048,160.0,0.010472,0.002104,0.008368,4.976450,True
4,Heston,cubic,4096,200.0,0.072264,0.013468,0.058796,5.365686,True
5,Heston,linear,2048,160.0,0.026912,0.004705,0.022208,5.720481,True


FFT speed check: Rust is faster in all tested configurations.


,model,interp,moneyness_bucket,n,max_abs_diff,p95_abs_diff,mean_abs_diff,max_rel_diff
0,BS,cubic,<0.85,10,1.421085e-13,1.357137e-13,4.973799e-14,2.842171e-06
1,BS,cubic,0.85-0.95,1,7.105427e-14,7.105427e-14,7.105427e-14,8.469038e-15
2,BS,cubic,0.95-1.05,3,1.421085e-13,1.385558e-13,8.289665e-14,4.476428e-15
3,BS,cubic,1.05-1.15,1,2.842171e-14,2.842171e-14,2.842171e-14,1.371029e-15
4,BS,cubic,>1.15,9,1.136868e-13,9.094947e-14,3.947460e-14,1.167821e-08
5,BS,linear,<0.85,10,1.705303e-13,1.193712e-13,3.552714e-14,8.113046e-16
6,BS,linear,0.85-0.95,1,2.842171e-14,2.842171e-14,2.842171e-14,3.379906e-15
7,BS,linear,0.95-1.05,3,1.421085e-13,1.335820e-13,7.815970e-14,5.709879e-15
8,BS,linear,1.05-1.15,1,2.842171e-14,2.842171e-14,2.842171e-14,1.369148e-15
9,BS,linear,>1.15,9,5.684342e-14,5.115908e-14,2.526374e-14,4.685596e-11


,model,interp,n_fft,b_fft,underlier_type,underlier,is_call,qty,expiry_days,moneyness,old,rust,abs_diff,rel_diff
111,Black,cubic,4096,200.0,forward,81.845889,False,5,180,0.569175,5.795187e-09,5.792948e-09,2.238210e-12,2.238210e-04
118,Black,cubic,4096,200.0,forward,128.433132,False,5,60,1.349563,7.432011e-08,7.431868e-08,1.421085e-12,1.912114e-05
35,Black,linear,2048,160.0,forward,123.376157,True,5,90,1.077811,4.414324e+01,4.414324e+01,1.413980e-12,3.203163e-14
104,Black,cubic,4096,200.0,forward,142.785174,True,5,90,2.259313,3.945140e+02,3.945140e+02,1.307399e-12,3.313947e-15
46,Black,linear,2048,160.0,forward,128.433132,False,5,60,1.349563,-2.115165e-02,-2.115165e-02,1.278977e-12,6.046699e-11
113,Black,cubic,4096,200.0,forward,141.666613,False,5,540,0.824120,3.731138e-08,3.731017e-08,1.207923e-12,3.237411e-05
107,Black,cubic,4096,200.0,forward,123.376157,True,5,90,1.077811,4.415237e+01,4.415237e+01,1.129763e-12,2.558782e-14
116,Black,cubic,4096,200.0,forward,143.554279,False,3,90,1.161230,3.187507e-08,3.187406e-08,1.012523e-12,3.176537e-05
108,Black,cubic,4096,200.0,forward,142.857815,False,3,540,1.030230,2.762437e-09,2.761466e-09,9.712231e-13,9.712231e-05
105,Black,cubic,4096,200.0,forward,152.282289,False,4,540,2.036629,3.862243e-08,3.862147e-08,9.663381e-13,2.502013e-05


BS: max_abs=0.00000000, p95_abs=0.00000000, max_rel_filtered=0.00000000
Black: max_abs=0.00000000, p95_abs=0.00000000, max_rel_filtered=0.00000000
Heston: max_abs=0.00000000, p95_abs=0.00000000, max_rel_filtered=0.00000000


## Interpretation

- Analytical sections should show near machine-precision parity.
- Monte Carlo sections should show close values with non-zero differences due to RNG/implementation differences.
- Throughput should generally favor Rust bindings, especially for batch and Monte Carlo workloads.

If a section diverges more than expected, inspect:

1. Option conventions (`call`/`put`, quantity, dates)
2. MC controls (`n_paths`, `n_steps`, `seed`, parallel flags)
3. Input dtypes and contiguity for NumPy APIs
